# Lesson 1 — Machine Learning Fundamentals in Databricks
**Goal:** understand one complete learning workflow before studying linear regression deeply in Lesson 2.

Run cells in order. Python and Markdown are separate. Outputs below are expected or illustrative, not results executed in your Databricks workspace. This notebook was syntax-checked outside Databricks; live Spark, Unity Catalog and MLflow operations require your workspace.

## Full learning roadmap
| Stage | Lessons, in order | Deliverable |
|---|---|---|
| 1: Foundations | 1. ML workflow; 2. Linear regression, with features, labels, splits and regression metrics; 3. Feature engineering, leakage, overfitting and underfitting | A reproducible regression baseline |
| 2: Regression | 4. Ridge; 5. Lasso; 6. Decision tree regression; 7. Random forest regression; 8. Gradient-boosted tree regression | Compare each new model with previous ones |
| 3: Classification | 9. Logistic regression; 10. Decision tree classification; 11. Random forest classification; 12. Gradient boosting classification; 13. Naive Bayes; 14. Linear support vector machines | Churn risk baseline; confusion matrix, accuracy, precision, recall, F1 and ROC-AUC |
| 4: Unsupervised learning | 15. K-means; 16. PCA; 17. Customer segmentation; 18. Anomaly detection | Segments with silhouette score and business interpretation |
| 5: Advanced models | 19. XGBoost; 20. LightGBM; 21. Time-series forecasting; 22. Recommendations; 23. Neural-network introduction | One notebook and one use case per model |
| 6: MLOps | 24. MLflow comparison and model lifecycle; 25. Reusable pipelines, functions/classes and pytest; 26. Jobs and CI/CD; 27. Monitoring/retraining and serving | Repeatable training and batch inference |
| 7: Final project | 28 onward: Customer churn ingestion → Delta → point-in-time features → training → evaluation → MLflow → model persistence → batch inference → tests → CI/CD → monitoring | End-to-end churn prediction system |

MLflow and production habits start now and become more detailed later. Spark's DataFrame ML API is `pyspark.ml`; avoid the older RDD-based `pyspark.mllib` for these notebooks. Ridge and Lasso are regularized linear models. Spark's LinearSVC is a linear classifier, not a general kernel SVM. Spark GBT classification is binary. Forecasting, LightGBM, isolation forests and neural networks need alternatives when native Spark ML does not supply the required algorithm: compatible XGBoost Spark integration, SynapseML where supported, or bounded NumPy arrays with single-node libraries. We will check environment support in each lesson and avoid pandas wherever possible.

**Pace:** one algorithm per lesson. Do not proceed to Lesson 2 until you confirm.

## Environment
Import this `.ipynb` into Databricks and attach serverless notebook compute. Use a recent supported base environment. Do not pip-install PySpark into Databricks. If MLflow is missing, add `mlflow` through notebook Environment dependencies and rerun from the top.

Free Edition has usage quotas, restricted outbound networking and serverless-only compute. This example downloads nothing. No RDD, SparkContext, DBFS mount, cache or persist calls are used. Spark ML support depends on the selected environment. Start with `ENGINE = "spark_ml"`; if an explicit unsupported-ML/API error occurs, change it to `"dataframe"` and rerun. That alternative learns the same unregularized, single-feature least-squares line through Spark aggregations; it is a teaching implementation, not a replacement for the general Spark ML library. Do not switch engines to hide data errors.

Persistence cells require a writable Unity Catalog schema. The notebook reads your current catalog and schema; change them to a writable location if needed. All output tables use a unique suffix, avoiding overwrite of existing tables.

Official references, checked 11 October 2026:
- https://docs.databricks.com/aws/en/getting-started/free-edition-limitations
- https://docs.databricks.com/aws/en/compute/serverless/limitations
- https://docs.databricks.com/aws/en/compute/serverless/dependencies
- https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html
- https://docs.databricks.com/aws/en/mlflow/tracking

## 1. Business problem
A delivery team needs to estimate journey time before dispatch. Input: distance in kilometres. Output: minutes. An estimate can support scheduling and customer ETAs. A real system also needs traffic, dispatch delay, weather, route and time-of-day features.

## 2. Algorithm introduction and fundamentals
**Machine learning:** learn a useful relationship from past examples, then apply it to new inputs. An algorithm is the learning procedure. A model is the learned result.

Traditional ETL applies rules you write. ML training uses historical inputs and known outcomes to estimate some of those rules. Prediction (also called inference) applies the saved result, much like a reusable transformation.

| Term | Meaning | Here |
|---|---|---|
| Observation | One example/row | One journey |
| Feature | Input available when predicting | distance_km |
| Label/target | Known answer during training | duration_min |
| Supervised learning | Learn from inputs and known answers | Predict journey duration |
| Regression | Predict a number | 32.5 minutes |
| Classification | Predict a category | Late / on-time |
| Unsupervised learning | Find structure without a target answer | Group customers by behaviour |
| Training | Estimate model parameters | Learn slope and intercept |
| Validation | Choose settings using separate examples | Compare intercept settings |
| Test | Final evaluation on untouched examples | Report final errors |

A straight-line model uses **predicted minutes = intercept + slope × distance**. The intercept is the predicted value at zero distance; the slope is extra minutes per kilometre. These are learned parameters. A hyperparameter is a setting chosen before fitting, such as whether to fit an intercept.

For an illustrative line `10 + 5 × distance`, a 4 km journey takes 30 predicted minutes. The algorithm chooses a line that minimizes the sum of squared training errors. It does not discover a law of nature or prove that distance causes every observed delay.

**Underfitting:** the model is too simple and misses useful patterns, often with poor training and validation scores. **Overfitting:** it captures training-specific noise and does poorly on new examples. A large train/validation gap is a warning, not a complete diagnosis.

**Data leakage:** the model uses information unavailable at prediction time, or test information influences training. Actual arrival time is leakage for pre-dispatch prediction. Computing imputation/scaling statistics on the entire dataset also leaks information: fit them on training only.

## 3. Real-world use case
Transport planning, demand forecasting and cost estimation use regression. Fraud flags and churn outcomes use classification. This lesson previews linear regression only; its detailed derivation and regularization follow later.

## 4. Dataset selection
We use **16 synthetic journeys**, not actual company data. Ten train rows contain small measurement noise; three validation and three test rows are held out in advance. It is small enough to inspect every record and avoids external permissions. It is unsuitable for establishing production accuracy. Later algorithm lessons will use freely available real datasets, checked for accessibility.

## 5. Data loading — Python cell 1
**What/why:** set the backend and create a typed Spark DataFrame so columns have predictable meanings.

- `functions as F` gives a short name for Spark SQL functions.
- `ENGINE` selects the training implementation, not a different algorithm.
- The comprehension builds ten training tuples; `range(1, 11)` includes 1 through 10. Alternating ±1 noise prevents a perfectly artificial fit.
- The extra tuples hold validation/test rows fixed for reproducible teaching.
- `createDataFrame(data, schema)` maps tuples to named, typed columns. `double` supports fractional values. `split` and `trip_id` are bookkeeping, not features.
- `show(16)` prints all rows; `orderBy` makes their display deterministic.

**Expected:** 16 rows; trip 1 has 1.0 km and 16.0 minutes; trip 2 has 2.0 km and 19.0 minutes. Test rows have IDs 14–16.
**Common errors:** integer Python values in explicitly double fields; tuples with the wrong number of fields; `spark` undefined outside an attached Databricks notebook.

In [ ]:
from pyspark.sql import functions as F
ENGINE = "spark_ml"  # Alternative: "dataframe"
assert ENGINE in {"spark_ml", "dataframe"}
rows = [(i, float(i), float(10 + 5*i + (1 if i % 2 else -1)), "train")
        for i in range(1, 11)]
rows += [(11, 2.5, 22.5, "validation"), (12, 5.5, 37.5, "validation"),
         (13, 8.5, 52.5, "validation"), (14, 3.5, 28.5, "test"),
         (15, 6.5, 41.5, "test"), (16, 9.5, 59.5, "test")]
df = spark.createDataFrame(rows,
    "trip_id int, distance_km double, duration_min double, split string")
df.orderBy("trip_id").show(16)

## 6. Exploratory data analysis — Python cell 2
**What/why:** inspect types and the training distribution before choosing preprocessing. Test labels are reserved for final evaluation.
- `printSchema()` shows names/types/nullability, not data values.
- `filter` selects training records. `describe` reports count, mean, standard deviation, minimum and maximum.
- `select` builds a projection; `isNull().cast("int")` maps missing values to 1 and others to 0; `sum` counts them; `alias` names the result.
**Expected:** training count 10; mean distance 5.5, mean duration 37.5; both null counts 0. Distance runs 1–10 km.
**Interpretation:** this dataset covers short trips only. Predictions for 100 km would be extrapolation, outside the observed range.
**Common errors:** column-name typos; string types producing unsuitable summaries. A null count does not detect NaN, so the cleaning cell handles that separately.

In [ ]:
df.printSchema()
train_raw = df.filter(F.col("split") == "train")
train_raw.describe("distance_km", "duration_min").show()
train_raw.select(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in ["distance_km", "duration_min"]
]).show()

## 7. Data cleaning — Python cell 3
**What/why:** apply predetermined validity rules and report excluded rows. These rules do not estimate anything from test labels. Missing labels cannot be invented for supervised training.
- `dropna(subset=...)` removes rows missing required fields.
- `filter` removes NaN, infinity, nonpositive distance and nonpositive duration; `&` means AND and `~` means NOT.
- `abs(value) != infinity` excludes both signs of infinity.
- `count` triggers execution. The duplicate check fails visibly instead of arbitrarily keeping conflicting records.
**Expected:** input 16, usable 16, excluded 0.
**Common errors:** using Python `and` for Spark expressions; missing parentheses; deleting so many records that a split becomes empty. For real data, write rejected rows and reasons to a quarantine Delta table.

In [ ]:
clean = df.dropna(subset=["trip_id", "distance_km", "duration_min", "split"])
clean = clean.filter(
    (~F.isnan("distance_km")) & (~F.isnan("duration_min")) &
    (F.abs(F.col("distance_km")) != float("inf")) &
    (F.abs(F.col("duration_min")) != float("inf")) &
    (F.col("distance_km") > 0) & (F.col("duration_min") > 0)
)
assert clean.groupBy("trip_id").count().filter(F.col("count") > 1).count() == 0
print({"input": df.count(), "usable": clean.count(),
       "excluded": df.count() - clean.count()})

## 8. Feature engineering — Python cell 4
**What/why:** declare the allowed predictor and create a reusable feature transformation.
- `feature_columns` excludes the label, ID and split, preventing obvious leakage.
- Spark ML expects a vector column: `VectorAssembler(inputCols, outputCol, handleInvalid)` turns distance 4.0 into vector `[4.0]`.
- `handleInvalid="error"` fails on invalid feature values instead of silently skipping them.
- `prepare_features(frame)` applies the assembler with `transform`. The alternative backend uses the numeric column directly.
This assembler learns no statistics, so defining it before splitting is safe. Learned imputers, encoders and scalers must be fitted inside the training split/pipeline.
**Expected:** no printed output; a reusable function is defined.
**Common errors:** unsupported Spark ML constructors in an older environment; a pre-existing `features` column; strings supplied as numeric features. Use the explicit DataFrame alternative only for environment incompatibility.

In [ ]:
feature_columns = ["distance_km"]
if ENGINE == "spark_ml":
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression
    assembler = VectorAssembler(inputCols=feature_columns,
                                outputCol="features", handleInvalid="error")

def prepare_features(frame):
    return assembler.transform(frame) if ENGINE == "spark_ml" else frame

## 9. Train/validation/test split — Python cell 5
**What/why:** assign separate jobs to separate records. Training learns coefficients, validation chooses the setting, test estimates final performance.
- `filter` uses the predetermined split values; `prepare_features` applies the same feature mapping to each split.
- `assert` stops execution if a split is unexpectedly empty.
**Expected:** train 10, validation 3, test 3.
For independent real observations, a seeded random split can be suitable. For future events, split chronologically. For repeated customer records, keep a customer in one split where appropriate. An arbitrary ID threshold is not a general splitting strategy.
**Common errors:** fitting before splitting, reusing test rows for tuning, or customers leaking across splits. Tiny random splits can be empty; fixed assignments avoid that here.

In [ ]:
train = prepare_features(clean.filter(F.col("split") == "train"))
validation = prepare_features(clean.filter(F.col("split") == "validation"))
test = prepare_features(clean.filter(F.col("split") == "test"))
for name, frame in [("train", train), ("validation", validation), ("test", test)]:
    n = frame.count()
    assert n > 0, f"Empty {name} split"
    print(name, n)

## 10. Model training — Python cell 6
**What/why:** learn slope and intercept only from training data; retain a portable representation for this one-feature model.
- `fit_line(frame, fit_intercept)` accepts a Spark DataFrame and a Boolean setting.
- Spark `LinearRegression`: `featuresCol` identifies the vector; `labelCol` identifies the answer; `predictionCol` names its output. `fitIntercept` controls the offset. `regParam=0.0` means no regularization penalty. `maxIter=100` is an iteration cap for solvers that iterate, not a promise of 100 passes. `fit` returns a trained model.
- `coefficients[0]` is the one slope; `intercept` is the offset. `float` makes portable Python numbers.
- The alternative uses `covar_pop(x,y)/var_pop(x)` for slope and `mean(y)-slope*mean(x)` for intercept. With the intercept disabled, slope is `sum(x*y)/sum(x*x)`. `agg` performs distributed reductions; `first` retrieves only one summary row, not the dataset. Assertions reject constant/zero features.
- Both return a dictionary containing the numbers and configuration. The Spark branch also keeps its native model.
**Expected:** slope approximately 4.939394; intercept approximately 10.333333. At zero distance the line predicts about 10.33 minutes, but zero is outside our training range and the intercept is not a measured delay.
**Common errors:** null vectors, unsupported backend, zero feature variance. The direct formula is teaching code; production libraries have stronger numerical handling for large-offset or nearly constant values.

In [ ]:
def fit_line(frame, fit_intercept=True):
    if ENGINE == "spark_ml":
        native = LinearRegression(featuresCol="features", labelCol="duration_min",
            predictionCol="prediction", fitIntercept=fit_intercept,
            regParam=0.0, maxIter=100).fit(frame)
        return {"slope": float(native.coefficients[0]),
                "intercept": float(native.intercept),
                "fit_intercept": fit_intercept, "native": native}
    x, y = F.col("distance_km"), F.col("duration_min")
    stats = frame.agg(F.avg(x).alias("mx"), F.avg(y).alias("my"),
        F.var_pop(x).alias("vx"), F.covar_pop(x, y).alias("cxy"),
        F.sum(x*y).alias("xy"), F.sum(x*x).alias("xx")).first()
    assert stats.vx is not None and stats.vx > 0, "Distance must vary"
    assert stats.xx > 0, "Distance sum of squares must be positive"
    slope = stats.cxy / stats.vx if fit_intercept else stats.xy / stats.xx
    intercept = stats.my - slope * stats.mx if fit_intercept else 0.0
    return {"slope": float(slope), "intercept": float(intercept),
            "fit_intercept": fit_intercept}

initial_model = fit_line(train)
print({k: v for k, v in initial_model.items() if k != "native"})

## 11. Prediction — Python cell 7
**What/why:** apply the learned relationship without retraining. We inspect validation predictions now; test remains untouched.
- `predict_line` accepts model numbers and a frame containing distance. It does not require labels.
- `withColumn` adds predicted minutes; `lit` embeds scalar coefficients into a distributed Spark expression.
- This formula equals the Spark model prediction for this exact single-feature linear model, and avoids collecting records to Python. It does not generalize to trees or complex preprocessing.
- `select`, `orderBy`, `show` display only relevant fields.
**Expected, rounded:** at 2.5, 5.5 and 8.5 km: 22.6818, 37.5000 and 52.3182 minutes.
**Common errors:** missing distance or a string distance. Preserve feature definitions/units when applying a saved model.

In [ ]:
def predict_line(model, frame):
    return frame.withColumn("prediction",
        F.lit(model["intercept"]) + F.lit(model["slope"]) * F.col("distance_km"))

validation_predictions = predict_line(initial_model, validation)
validation_predictions.select("trip_id", "distance_km", "duration_min", "prediction").orderBy("trip_id").show()

## 12. Model evaluation — Python cell 8
**Metrics:** error = actual − prediction. For actual `[10,20,30]` and predicted `[12,18,33]`, errors are `[-2,2,-3]`.

| Metric | Calculation in the example | Use |
|---|---|---|
| MAE | (2+2+3)/3 = 2.333 minutes | Average absolute miss; easy to explain |
| MSE | (4+4+9)/3 = 5.667 minutes² | Penalizes large misses; common optimization objective |
| RMSE | sqrt(5.667) = 2.380 minutes | Penalizes large misses, in original units |
| R² | 1−17/200 = 0.915 | Compares squared error with predicting the evaluation-set mean |

Lower is better for MAE/MSE/RMSE. R²=1 is perfect; 0 matches that mean reference; negative is worse. R² is undefined for a constant target under this definition. R² is not a percentage accuracy.

**What/why:** a reusable evaluator gives consistent metrics and a train-mean baseline provides an honest simple competitor.
- `regression_metrics` uses `agg` to calculate absolute/squared error sums and target variance. `var_pop * n` equals total squared distance from the target mean.
- `asDict` converts the single aggregated Row into a dictionary. `**0.5` is square root; the conditional avoids division by zero for R².
- `validation` metrics guide model selection; this is not final test performance.
- The baseline predicts the training average for every validation row. It learns only one constant, unlike the distance-based model.
**Expected:** validation MAE ≈0.1212, RMSE ≈0.1485. Baseline predicts 37.5, validation RMSE ≈12.2474. Scores are unusually good because our synthetic dataset was constructed to be nearly linear.
**Common errors:** averaging signed errors (cancellation), confusing MSE units with minutes, using classification accuracy for regression, or missing labels when evaluating.

In [ ]:
def regression_metrics(predictions):
    error = F.col("duration_min") - F.col("prediction")
    stats = predictions.agg(F.count("duration_min").alias("n"),
        F.avg(F.abs(error)).alias("mae"), F.avg(error*error).alias("mse"),
        F.sum(error*error).alias("sse"),
        F.var_pop("duration_min").alias("target_var")).first().asDict()
    assert stats["n"] > 0, "No labelled predictions"
    sst = stats["target_var"] * stats["n"]
    return {"mae": stats["mae"], "mse": stats["mse"],
            "rmse": stats["mse"] ** 0.5,
            "r2": 1 - stats["sse"] / sst if sst > 0 else None}

train_mean = train.agg(F.avg("duration_min").alias("mean")).first()["mean"]
print("validation model:", regression_metrics(validation_predictions))
print("validation baseline:", regression_metrics(
    validation.withColumn("prediction", F.lit(train_mean))))

## 13. Hyperparameter tuning — Python cell 9
**What/why:** compare two settings using validation only. `fitIntercept` is a small, understandable tuning demonstration; it is not a full search strategy for production.
- `for` trains one candidate per Boolean setting.
- `candidates` stores settings, models and validation scores; `min(..., key=...)` selects the lowest RMSE.
- We keep the selected model trained on the original training split for clarity. A later lesson will refit on train+validation after settings are frozen.
- Only after selection do we evaluate the untouched test set, exactly once in this workflow. Repeatedly redesigning based on these test scores would compromise its independence.
**Expected:** `True` wins; false has validation RMSE about 4.1142 versus 0.1485. Final test predictions ≈27.6212, 42.4394, 57.2576. Test MAE ≈1.3535 and RMSE ≈1.4928; baseline RMSE ≈13.9164. Tiny samples cannot support reliable production claims.
**Common errors:** selecting based on test RMSE, treating training score as future performance, or assuming validation wins always transfer to test.

In [ ]:
candidates = []
for setting in [True, False]:
    candidate = fit_line(train, fit_intercept=setting)
    metrics = regression_metrics(predict_line(candidate, validation))
    candidates.append({"model": candidate, "metrics": metrics})
    print("fitIntercept", setting, "validation", metrics)
selected = min(candidates, key=lambda item: item["metrics"]["rmse"])
best_model = selected["model"]
test_predictions = predict_line(best_model, test)
test_metrics = regression_metrics(test_predictions)
baseline_test_metrics = regression_metrics(test.withColumn("prediction", F.lit(train_mean)))
test_predictions.select("trip_id", "duration_min", "prediction").orderBy("trip_id").show()
print("test model:", test_metrics)
print("test baseline:", baseline_test_metrics)

## 14. MLflow experiment tracking — Python cell 10
**What/why:** save the experiment history: settings, validation metrics, final metrics and portable model definition. An experiment groups runs; a run records one trial.
- `set_tracking_uri("databricks")` uses the workspace tracking service.
- `current_user()` gets the notebook identity. The experiment path is inside that user's workspace home. `set_experiment` selects or creates it.
- `start_run` opens/closes each run via a context manager. `log_params` logs chosen settings; `log_metrics` logs numeric scores.
- Two candidate runs have validation metrics. A separately named selected-model run records final test metrics, baseline metrics, and training parameters.
- `log_dict` stores a JSON artifact containing the exact one-feature model specification; this is not an MLflow model flavor or registered model.
- `run.info.run_id` identifies the selected run for lineage.
**Expected:** three runs appear in the experiment. Compare candidate runs by `validation_rmse`; do not choose models using the selected run's test scores.
**Common errors:** missing mlflow dependency, no workspace-home permission, or an already active run. Finish the unrelated run first; do not silently terminate it. Change the experiment path to an existing writable workspace directory when required.

In [ ]:
import mlflow
mlflow.set_tracking_uri("databricks")
user_name = spark.sql("SELECT current_user() AS u").first()["u"]
experiment_path = f"/Users/{user_name}/lesson_01_delivery_fundamentals"
mlflow.set_experiment(experiment_path)
for item in candidates:
    with mlflow.start_run(run_name=f"candidate_intercept_{item['model']['fit_intercept']}"):
        mlflow.log_params({"engine": ENGINE, "fit_intercept": item["model"]["fit_intercept"],
                           "feature": "distance_km", "dataset": "synthetic_delivery_v1"})
        mlflow.log_metrics({f"validation_{k}": v for k, v in item["metrics"].items() if v is not None})
with mlflow.start_run(run_name="selected_model_final_test") as run:
    mlflow.log_params({"engine": ENGINE, "fit_intercept": best_model["fit_intercept"],
                       "slope": best_model["slope"], "intercept": best_model["intercept"],
                       "dataset": "synthetic_delivery_v1", "spark_version": spark.version})
    mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items() if v is not None})
    mlflow.log_metrics({f"baseline_test_{k}": v for k, v in baseline_test_metrics.items() if v is not None})
    mlflow.log_dict({"model_type": "single_feature_linear_regression", "feature": "distance_km",
        "label": "duration_min", "slope": best_model["slope"],
        "intercept": best_model["intercept"]}, "model/linear_model.json")
    selected_run_id = run.info.run_id
print("Experiment:", experiment_path)
print("Selected run:", selected_run_id)

## 15. Model saving and loading — Python cell 11
**What/why:** persist this simple model as a one-row Delta table and load it for later inference. Two coefficients plus the feature/unit contract completely specify this particular prediction function. This is a portable custom format, **not** Spark PipelineModel serialization and **not** Unity Catalog model registration.
- `current_catalog/current_schema` locate the active namespace. Override the returned strings if that namespace is read-only.
- `quote_identifier` doubles embedded backticks and quotes SQL identifier parts safely.
- `uuid4().hex[:10]` creates a unique suffix. Keep the printed table names for a future session.
- `saveAsTable` writes managed Delta tables; `errorifexists` prevents destructive overwrite. We persist source records, coefficients and lineage.
- `spark.table` reloads by name. `first` retrieves the one-row model. `assert` verifies feature/label expectations. In a new notebook, set `model_table` to the printed name before reading it.
**Expected:** two new tables, and reloaded slope ≈4.939394 / intercept ≈10.333333.
**Common errors:** no CREATE TABLE/USE permissions, read-only sample catalog, or an incorrect three-part name. Select a writable catalog/schema in Catalog Explorer. Training and evaluation do not depend on Delta write privileges, but durable saving does.
For general Spark pipelines we will use native pipeline persistence and a supported MLflow flavor; model registration additionally requires a compatible model artifact, signature and Unity Catalog permissions. A logged JSON artifact alone is not deployable through normal model serving.

In [ ]:
from uuid import uuid4
namespace = spark.sql("SELECT current_catalog() AS c, current_schema() AS s").first()
CATALOG, SCHEMA = namespace.c, namespace.s  # Change to a writable namespace if needed.
def quote_identifier(value):
    return "`" + value.replace("`", "``") + "`"
prefix = quote_identifier(CATALOG) + "." + quote_identifier(SCHEMA)
run_suffix = uuid4().hex[:10]
source_table = prefix + "." + quote_identifier("ml_lesson01_source_" + run_suffix)
model_table = prefix + "." + quote_identifier("ml_lesson01_model_" + run_suffix)
prediction_table = prefix + "." + quote_identifier("ml_lesson01_predictions_" + run_suffix)
clean.write.format("delta").mode("errorifexists").saveAsTable(source_table)
model_record = spark.createDataFrame([
    (best_model["slope"], best_model["intercept"], "distance_km", "duration_min", selected_run_id)
], "slope double, intercept double, feature string, label string, mlflow_run_id string")
model_record.write.format("delta").mode("errorifexists").saveAsTable(model_table)
saved = spark.table(model_table).first()
assert saved.feature == "distance_km" and saved.label == "duration_min"
loaded_model = {"slope": saved.slope, "intercept": saved.intercept}
print("Source table:", source_table)
print("Model table:", model_table)
print("Loaded model:", loaded_model)

## 16. Batch prediction — Python cell 12
**What/why:** score several new journeys with the reloaded model, without labels or another training run.
- `createDataFrame` creates three new inputs. `predict_line` applies coefficients using distributed Spark expressions.
- `withColumn` attaches the MLflow run ID and scoring timestamp for traceability.
- A Delta output table can be consumed by SQL, a dashboard or a downstream job.
**Expected, rounded:** 4 km →30.0909 minutes; 7 km →44.9091; 10 km →59.7273. These are estimates, not guaranteed arrival times.
**Common errors:** passing metres as kilometres, changing preprocessing, null/negative new distances, or duplicate scoring on retries. The sample inputs are valid; a production batch must validate them and quarantine failures before scoring. Define a stable key such as trip ID plus model version and use an idempotent write strategy.
This is offline DataFrame scoring in a notebook/job; it does not require a hosted model-serving endpoint.

In [ ]:
new_trips = spark.createDataFrame([(101, 4.0), (102, 7.0), (103, 10.0)],
                                  "trip_id int, distance_km double")
batch_predictions = (predict_line(loaded_model, new_trips)
    .withColumn("mlflow_run_id", F.lit(saved.mlflow_run_id))
    .withColumn("scored_at", F.current_timestamp()))
batch_predictions.write.format("delta").mode("errorifexists").saveAsTable(prediction_table)
spark.table(prediction_table).orderBy("trip_id").show(truncate=False)
print("Prediction table:", prediction_table)

## 17. Production considerations
| Concern | Production action |
|---|---|
| Data lineage | Record source table versions, feature definitions, training cutoff, code commit and environment dependencies |
| Data quality | Reject/quarantine invalid records; validate units, ranges, schema and keys for training and inference |
| Leakage | Point-in-time feature joins; exclude post-outcome fields; fit preprocessing on training only |
| Evaluation | Compare against simple baselines; time/group-aware splits; evaluate by route, distance band and relevant business slices |
| Reliability | More data, uncertainty analysis and an untouched test set; never use this 16-row result as a production guarantee |
| Feature drift | Watch changes in distances, routes, missing rates and input distributions |
| Performance drift | Once actual durations arrive, join them to predictions and monitor MAE/RMSE over time |
| Retraining | Trigger on validated performance degradation or a schedule; evaluate a candidate before promotion; retain rollback model |
| Jobs | Separate training, validation and scoring tasks; enforce quality gates before downstream writes |
| CI/CD | Version modules/configs, pin supported dependencies, test transformations and contracts with pytest; integration-test a small job before deployment |
| Model lifecycle | Track runs; save full preprocessing/model together for richer algorithms; register compatible models when supported and permitted |
| Serving | Batch is sufficient here; online serving requires a supported packaged model, endpoint permissions, input contract and latency/availability checks |

The saved coefficients are sufficient only because preprocessing here is an identity mapping of one numeric feature. A model with learned imputation, categorical encoding or multiple features must preserve those transformations and feature order too.

Free Edition is for learning/non-commercial use, has quotas and no production SLA. Hosted serving has additional feature limits; offline notebook predictions are a separate workflow. We will demonstrate production architecture as a design, and identify which pieces need a suitable paid workspace.

### Debugging scenario
**Problem:** a model appears nearly perfect on historical data but fails when predicting new dispatches. The feature list includes `actual_arrival_timestamp`.
**Diagnosis:** that timestamp exists only after the journey ends. The training data leaks the answer.
**Fix:** remove post-dispatch information, rebuild point-in-time features, split chronologically, refit preprocessing on training only, and reevaluate against an untouched future period. A lower but honest score is more useful.

### Three practical exercises
1. Predict 6 km using the learned slope/intercept. Expected ≈39.9697 minutes. Explain why it is not exactly 40.
2. Add a null-distance record to `rows`. Rerun loading/cleaning and explain why usable rows stay 16 while input becomes 17. Do not replace a missing label with zero.
3. Add 10 km →90 minutes to the **training** set using a new ID. Rerun and compare validation RMSE. Explain how an unusual training record can move the line. Do not select changes using test results.

### Five interview questions with answers
1. **What distinguishes a feature from a label?** A feature is an input known at prediction time; a label is the outcome learned during training.
2. **Training versus prediction?** Training estimates parameters from examples. Prediction applies those parameters to new inputs.
3. **Why use validation and test splits?** Validation selects settings. A held-out test set provides a final assessment after those choices.
4. **What is leakage?** Information from the future, target, or held-out evaluation set influences the model improperly.
5. **Why compare with a baseline?** A complex model must improve enough over a simple alternative to justify its cost and maintenance.

### Mini assignment
Create at least 30 synthetic delivery rows containing distance and duration, including realistic noise. Label them explicitly as synthetic. Allocate 20 training, 5 validation and 5 test records before inspecting results. Run this workflow and submit: split counts, learned equation, validation RMSE, final test MAE/RMSE, baseline comparison and two limitations. Do not introduce another algorithm yet.

### Short quiz — answer before reading the key
1. Which is the label here: A) trip ID B) distance C) actual duration D) split name?
2. Which data chooses `fitIntercept`: A) training score only B) validation C) test D) new unlabelled trips?
3. What does MAE=3 minutes mean: A) every prediction is off by 3 B) 97% accurate C) average absolute error is 3 D) model trains in 3 minutes?

<details><summary>Quiz answer key</summary>1: C. Duration is the known outcome. 2: B. Validation guides setting selection. 3: C. Individual errors can be smaller or larger.</details>

### Understanding check — stop here
Explain in your own words:
1. What does `fit()` learn, and what happens during prediction?
2. Why are `trip_id`, `split`, and `duration_min` excluded from the feature list?
3. Why must we not tune using test results?
4. Is this workflow regression, classification or clustering? Why?
5. Can these 16 synthetic records prove that the model is ready for real deliveries?

Tell me which cells you ran and share any error text. We will proceed to Lesson 2 only after your confirmation.